In [1]:
!pip install torch transformers librosa numpy

   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/109.3 MB ? eta -:--:--
   ---------------------------------------- 1.0/109.3 MB 3.2 MB/s eta 0:00:35
    --------------------------------------- 1.8/109.3 MB 3.3 MB/s eta 0:00:33
    --------------------------------------- 2.6/109.3 MB 3.2 MB/s eta 0:00:34
   - -------------------------------------- 2.9/109.3 MB 3.0 MB/s eta 0:00:35
   - -------------------------------------- 3.1/109.3 MB 2.8 MB/s eta 0:00:39
   - -------------------------------------- 3.9/109.3 MB 2.7 MB/s eta 0:00:39
   - -------------------------------------- 5.0/109.3 MB 3.0 MB/s eta 0:00:35
   -- ------------------------------------- 5.8/109.3 MB 3.2 MB/s eta 0:00:33
   -- ------------------------------------- 6.8/109.3 MB 3.4 MB/s eta 0:00:31
   -- ------------------------------------- 7.3/109.3 MB 3.3 MB/s eta 0:00:31
   -- ------------------------------------- 7.9/109.3 MB 3.2 MB/s eta 0:00:32


In [4]:
# Requires: librosa
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import librosa
import torch
import numpy as np

model_id = "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3"
model = AutoModelForAudioClassification.from_pretrained(model_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True)
id2label = model.config.id2label


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [5]:
def preprocess_audio(audio_path, feature_extractor, max_duration=30.0):
    audio_array, sampling_rate = librosa.load(audio_path, sr=None)
    
    max_length = int(feature_extractor.sampling_rate * max_duration)
    if len(audio_array) > max_length:
        audio_array = audio_array[:max_length]
    else:
        audio_array = np.pad(audio_array, (0, max_length - len(audio_array)))

    inputs = feature_extractor(
        audio_array,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    return inputs


In [6]:
def predict_emotion(audio_path, model, feature_extractor, id2label, max_duration=30.0):
    inputs = preprocess_audio(audio_path, feature_extractor, max_duration)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_id]
    
    return predicted_label


In [19]:
audio_path = "angry.mp3"

predicted_emotion = predict_emotion(audio_path, model, feature_extractor, id2label)
print(f"Predicted Emotion: {predicted_emotion}")

Predicted Emotion: angry
